### Chatbot evaluation


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [5]:
from langsmith import Client

client = Client()  # RESPONSIBLE FOR CREATING DATASETS AND EXAMPLES ON LANGSMITH PLATFORM

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['71c7590e-8a86-4313-9e8f-526edd25003c',
  'bf76f141-6408-4a2e-b82a-2d74c52cb1a8',
  '498ffb20-c178-41a6-ad73-62cdb4be4c82',
  '06e4e57f-ea43-41c0-a64d-e221715d0793',
  'db332215-bb83-4f67-896f-96869c88eae6'],
 'count': 5,
 'as_of': '2026-09-03T11:07:39.868060024Z'}

In [15]:
import os
import openai
from langsmith import wrappers

openai_client = wrappers.wrap_openai(
    openai.OpenAI(
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1"
    )
)

eval_instructions = """
You are an expert professor specialized in grading
students' answers to questions.
"""

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    user_content = f"""
You are grading the following question:

{inputs['question']}

Here is the real answer:

{reference_outputs['answer']}

You are grading the following predicted answer:

{outputs['response']}

Respond with exactly CORRECT or INCORRECT.

Grade:
"""

    response = openai_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": eval_instructions
            },
            {
                "role": "user",
                "content": user_content
            }
        ]
    ).choices[0].message.content

    return response.strip().upper() == "CORRECT"

In [16]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [21]:
# Run Evaluations
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "openai/gpt-oss-120b", instructions: str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [22]:
# Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [ ]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4o-mini-chatbot"  # u can name anything you want, this is just for your reference
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-ca0264ee' at:
https://smith.langchain.com/o/7f1d6b87-b031-4541-87bd-ad0215cbf98e/datasets/e3cce601-ffa4-467a-9f87-cff58fa85fe9/compare?selectedSessions=d0ef09fd-7573-4903-bc3e-3f17c02e4488




5it [00:07,  1.46s/it]


In [ ]:
# check the results at: https://smith.langchain.com/o/7f1d6b87-b031-4541-87bd-ad0215cbf98e/datasets/e3cce601-ffa4-467a-9f87-cff58fa85fe9/compare?selectedSessions=d0ef09fd-7573-4903-bc3e-3f17c02e4488

In [25]:
# check the results for some other model

def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"], model="openai/gpt-oss-20b")}

In [26]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai/gpt-oss-20b"
)

View the evaluation results for experiment: 'openai/gpt-oss-20b-766ebdd2' at:
https://smith.langchain.com/o/7f1d6b87-b031-4541-87bd-ad0215cbf98e/datasets/e3cce601-ffa4-467a-9f87-cff58fa85fe9/compare?selectedSessions=fddc606b-a360-485d-9177-be3d17e78859




5it [00:25,  5.07s/it]
